In [ ]:

import os, sys, glob, json, time, subprocess, shutil
T0 = time.time(); TRAN_GIO = 11.0
MN, MX = 200704, 1003520        # cỡ ảnh đã chốt ở phép B (20/8): 1.272 token. KHÔNG đổi.
subprocess.run("pip uninstall -y -q torchao", shell=True)
WS = "/kaggle/working"; OUT = f"{WS}/out"; os.makedirs(OUT, exist_ok=True)
TOM = open(f"{OUT}/tom_tat.txt", "a")
def ghi(*a):
    s = f"[{(time.time()-T0)/3600:5.2f} h] " + " ".join(map(str, a)); print(s, flush=True); TOM.write(s + "\n"); TOM.flush()
mp = glob.glob("/kaggle/input/**/harness/score_run.py", recursive=True); assert mp
SRC = os.path.dirname(os.path.dirname(mp[0]))
if os.path.exists(f"{WS}/harness"): shutil.rmtree(f"{WS}/harness")
shutil.copytree(f"{SRC}/harness", f"{WS}/harness")
SR = f"{WS}/harness/score_run.py"
t = open(SR, encoding="utf-8").read(); assert "VENUS_MIN_PIXELS" in t[t.index("class UIVenus"):], "harness chưa đọc cỡ ảnh"
im = sorted(glob.glob("/kaggle/input/**/ep18171_s1.png", recursive=True)); assert im, "thiếu ảnh (Add Data thesis-score)"
TEST = f"{WS}/harness/dg1_cache/test_ac"
if not os.path.exists(f"{TEST}/images"): os.symlink(os.path.dirname(im[0]), f"{TEST}/images")
PR = os.path.dirname(sorted(glob.glob("/kaggle/input/**/preds/preds_m1.jsonl", recursive=True))[0])
REF = sorted(glob.glob("/kaggle/input/**/ref/score_venus_s1_2532_raw.jsonl", recursive=True))[0]
ghi("harness", SR, "· ảnh", os.path.dirname(im[0]), "· preds", PR)
env = {**os.environ, "VENUS_MIN_PIXELS": str(MN), "VENUS_MAX_PIXELS": str(MX), "PYTHONUNBUFFERED": "1",
       "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
def doc(f):
    D = {}
    if os.path.exists(f):
        for l in open(f, encoding="utf-8"):
            try: o = json.loads(l); D[(o["episode_id"], o["step_id"])] = o
            except Exception: pass
    return D
def chay(ten):
    pf = f"{PR}/preds_{ten}.jsonl"; n = sum(1 for _ in open(pf))
    raw = f"{OUT}/score_{ten}_raw.jsonl"
    log = open(f"{OUT}/{ten}.log", "a")
    p = subprocess.Popen([sys.executable, "-u", SR, "--mode", "score", "--grounder", "uivenus", "--preds", pf,
                          "--n", str(n), "--out", f"{OUT}/score_{ten}.json"], stdout=log, stderr=subprocess.STDOUT, cwd=WS, env=env, start_new_session=True)
    while p.poll() is None:
        time.sleep(120)
        d = open(f"{OUT}/{ten}.log").read().strip().splitlines()[-1:] or [""]
        print(f"  {(time.time()-T0)/3600:5.2f} h · {ten}: {len(doc(raw))}/{n} · {d[0][:100]}", flush=True)
        if (time.time() - T0) / 3600 > TRAN_GIO: ghi("chạm trần giờ — dừng"); p.terminate(); time.sleep(20); break
    ghi(ten, "mã thoát", p.poll(), "·", len(doc(raw)), "/", n)
    return doc(raw)

# 0. phép kiểm tái lập: 40 bước S1 đã có kết quả UI-Venus (phép B) phải ra ĐÚNG toạ độ cũ
ref = doc(REF); kiem = chay("kiem")
so = [k for k in kiem if k in ref]
lech = sum(1 for k in so if (kiem[k].get("pred_xy") != ref[k].get("pred_xy")))
ghi(f"KIỂM TÁI LẬP: {len(so)} bước · toạ độ lệch {lech} (cần 0). Lệch ⇒ môi trường khác lượt phép B, KẾT QUẢ SAU ĐÂY KHÔNG DÙNG ĐƯỢC.")
for ten in ("m1",):
    if TRAN_GIO - (time.time() - T0) / 3600 < 0.5: ghi("bỏ qua", ten); break
    chay(ten)
subprocess.run(f"cd {WS} && zip -qr venus_ck.zip out", shell=True)
ghi("KẾT THÚC")
